In [ ]:
# ============================================
# MLflow Lab2 - End-to-End ML Pipeline
# Dataset: Wine Classification
# ============================================

import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings("ignore")

# --- Step 1: Load Data ---
print("STEP 1: Load Data")
df = pd.read_csv("data/wine_classification.csv")
print(f"Dataset shape: {df.shape}")
print(f"Target distribution:\n{df['target'].value_counts()}")

# --- Step 2: Data Splitting ---
print("\nSTEP 2: Data Splitting")
from sklearn.model_selection import train_test_split

X = df.drop(["target"], axis=1)
y = df["target"]
X_train, X_rem, y_train, y_rem = train_test_split(X, y, train_size=0.6, random_state=123)
X_val, X_test, y_val, y_test = train_test_split(X_rem, y_rem, test_size=0.5, random_state=123)
print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

# --- Step 3: Baseline Model ---
print("\nSTEP 3: Baseline Model - Random Forest")
import mlflow
import mlflow.pyfunc
import mlflow.sklearn
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from mlflow.models.signature import infer_signature
from mlflow.utils.environment import _mlflow_conda_env
import cloudpickle

class SklearnModelWrapper(mlflow.pyfunc.PythonModel):
    def __init__(self, model):
        self.model = model
    def predict(self, context, model_input):
        return self.model.predict(model_input)

with mlflow.start_run(run_name="untuned_random_forest"):
    n_estimators = 10
    model = RandomForestClassifier(n_estimators=n_estimators, random_state=123)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average="weighted")
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("f1_score", f1)
    wrappedModel = SklearnModelWrapper(model)
    signature = infer_signature(X_train, wrappedModel.predict(None, X_train))
    conda_env = _mlflow_conda_env(
        additional_conda_deps=None,
        additional_pip_deps=[f"cloudpickle=={cloudpickle.__version__}", f"scikit-learn=={sklearn.__version__}"],
        additional_conda_channels=None,
    )
    mlflow.pyfunc.log_model("random_forest_model", python_model=wrappedModel, conda_env=conda_env, signature=signature)
    print(f"Baseline Accuracy: {acc:.4f}, F1: {f1:.4f}")

# Feature Importance
feat_imp = pd.DataFrame(model.feature_importances_, index=X_train.columns.tolist(), columns=["importance"])
print("Top 5 Features:")
print(feat_imp.sort_values("importance", ascending=False).head())

# --- Step 4: Register Model ---
print("\nSTEP 4: Register Model")
run_id = mlflow.search_runs(filter_string='tags.mlflow.runName = "untuned_random_forest"').iloc[0].run_id
model_name = "wine_classifier"
model_version = mlflow.register_model(f"runs:/{run_id}/random_forest_model", model_name)
time.sleep(10)
print(f"Registered '{model_name}' version {model_version.version}")

# --- Step 5: Promote to Production ---
print("\nSTEP 5: Promote to Production")
from mlflow.tracking import MlflowClient
client = MlflowClient()
client.transition_model_version_stage(name=model_name, version=model_version.version, stage="Production")
prod_model = mlflow.pyfunc.load_model(f"models:/{model_name}/production")
print(f"Production Model Accuracy: {accuracy_score(y_test, prod_model.predict(X_test)):.4f}")

# --- Step 6: Hyperparameter Tuning ---
print("\nSTEP 6: XGBoost + Hyperopt Tuning (10 trials)")
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
from hyperopt.pyll import scope
import mlflow.xgboost
import xgboost as xgb

search_space = {
    "max_depth": scope.int(hp.quniform("max_depth", 4, 15, 1)),
    "learning_rate": hp.loguniform("learning_rate", -3, 0),
    "reg_alpha": hp.loguniform("reg_alpha", -5, -1),
    "reg_lambda": hp.loguniform("reg_lambda", -6, -1),
    "objective": "multi:softmax",
    "num_class": 3,
    "seed": 123,
}

def train_model(params):
    mlflow.xgboost.autolog()
    with mlflow.start_run(nested=True):
        train = xgb.DMatrix(data=X_train, label=y_train)
        validation = xgb.DMatrix(data=X_val, label=y_val)
        booster = xgb.train(params=params, dtrain=train, num_boost_round=50,
                            evals=[(validation, "validation")], early_stopping_rounds=10, verbose_eval=False)
        val_preds = booster.predict(validation)
        val_acc = accuracy_score(y_val, val_preds)
        mlflow.log_metric("accuracy", val_acc)
        signature = infer_signature(X_train, booster.predict(train))
        mlflow.xgboost.log_model(booster, "model", signature=signature)
        return {"status": STATUS_OK, "loss": -1 * val_acc}

trials = Trials()
with mlflow.start_run(run_name="xgboost_models"):
    best_params = fmin(fn=train_model, space=search_space, algo=tpe.suggest, max_evals=10, trials=trials)
print(f"Best params: {best_params}")

# --- Step 7: Promote Best Model ---
print("\nSTEP 7: Promote Best XGBoost Model")
best_run = mlflow.search_runs(order_by=["metrics.accuracy DESC"]).iloc[0]
print(f"Best Accuracy: {best_run['metrics.accuracy']:.4f}")
new_model_version = mlflow.register_model(f"runs:/{best_run.run_id}/model", model_name)
time.sleep(10)
client.transition_model_version_stage(name=model_name, version=model_version.version, stage="Archived")
client.transition_model_version_stage(name=model_name, version=new_model_version.version, stage="Production")
print(f"Model version {new_model_version.version} promoted to Production!")

# --- Step 8: Final Evaluation ---
print("\nSTEP 8: Final Evaluation")
final_model = mlflow.pyfunc.load_model(f"models:/{model_name}/production")
print(f"Final Accuracy: {accuracy_score(y_test, final_model.predict(X_test)):.4f}")

# --- Step 9: Serve ---
print("\nSTEP 9: To serve, run in terminal:")
print(f"mlflow models serve --env-manager=local -m models:/{model_name}/production -h 0.0.0.0 -p 5001")
print("\nDONE!")